## **Homework**

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [4]:
df =  pd.read_csv(
    "smile-annotations-final.csv",
    names=['id','text','category']
)

In [5]:
df = df[df['category'].isin(['happy', 'angry', 'disgust', 'sad', 'not-relevant', 'surprise'])]

In [6]:
df.head()

,id,text,category
1,614484565059596288,Dorian Gray with Rainbow Scarf #LoveWins (from...,happy
2,614746522043973632,@SelectShowcase @Tate_StIves ... Replace with ...,happy
3,614877582664835073,@Sofabsports thank you for following me back. ...,happy
4,611932373039644672,@britishmuseum @TudorHistory What a beautiful ...,happy
5,611570404268883969,@NationalGallery @ThePoldarkian I have always ...,happy


In [7]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['category'])

num_classes = len(label_encoder.classes_)
label_encoder.classes_

array(['angry', 'disgust', 'happy', 'not-relevant', 'sad', 'surprise'],
      dtype=object)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [10]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

train_encodings = tokenize(X_train)
val_encodings = tokenize(X_val)

In [11]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item



In [12]:
train_dataset = TextDataset(train_encodings, y_train)
val_dataset   = TextDataset(val_encodings, y_val)


In [15]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)


In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes
)

model.to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [14]:
optimizer =torch.optim.AdamW(model.parameters(), lr=2e-5)

In [16]:
epochs = 3

for epoch in range(epochs):
    # Train
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Train"):
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    preds, labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - Val"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            val_loss += outputs.loss.item()
            predictions = torch.argmax(outputs.logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    acc = accuracy_score(labels, preds)


print(f"Epoch {epoch+1} | Train loss: {avg_train_loss:.4f} | Val loss: {avg_val_loss:.4f} | Val acc: {acc:.4f}")


Epoch 3 - Val: 100%|██████████| 19/19 [00:00<00:00, 35.24it/s]

Epoch 3 | Train loss: 0.3310 | Val loss: 0.4659 | Val acc: 0.8519


In [17]:
model.save_pretrained("smile_classifier")
tokenizer.save_pretrained("smile_classifier")

('smile_classifier/tokenizer_config.json',
 'smile_classifier/special_tokens_map.json',
 'smile_classifier/vocab.txt',
 'smile_classifier/added_tokens.json',
 'smile_classifier/tokenizer.json')

In [21]:
text = "I feel very happy today!"

inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()
label_encoder.inverse_transform([pred])

array(['happy'], dtype=object)

In [23]:
from huggingface_hub import login
login()

Token has not been saved to git credential helper.


In [24]:
from huggingface_hub import whoami
whoami()


{'type': 'user',
 'id': '67a9e600e7be9759e2fd1a70',
 'name': 'Sanjarbek1024',
 'fullname': 'SANJARBEK GULOMJONOV',
 'email': 'gsanjarbek1024@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1769904000,
 'isPro': False,
 'avatarUrl': '/avatars/60d2b31f9b8c1ffbe72564193c74136f.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'fine-tune',
   'role': 'write',
   'createdAt': '2026-01-20T08:12:35.845Z'}}}

In [25]:
repo_id = "Sanjarbek1024/smile-text-classifier"

In [26]:
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/smile-text-classifier/commit/e5df4afc5d021b21a4196123aa285299a2f92f9d', commit_message='Upload tokenizer', commit_description='', oid='e5df4afc5d021b21a4196123aa285299a2f92f9d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/smile-text-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/smile-text-classifier'), pr_revision=None, pr_num=None)

In [27]:
%%writefile README.md
---
language: en
tags:
- text-classification
- emotion-detection
- distilbert
- pytorch
---

# SMILE Text Classifier

This model is fine-tuned on the SMILE dataset for emotion classification.

## Labels
- angry
- disgust
- happy
- not-relevant
- sad
- surprise

## Base model
- distilbert-base-uncased

## Training
- Epochs: 3
- Batch size: 16
- Optimizer: AdamW
- Learning rate: 2e-5


Overwriting README.md


In [28]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_id
)

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/smile-text-classifier/commit/0a97548e088a288711f44f2903a1f5072e0fdb5d', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='0a97548e088a288711f44f2903a1f5072e0fdb5d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/smile-text-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/smile-text-classifier'), pr_revision=None, pr_num=None)

In [29]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=repo_id
)

classifier("I am extremely happy today!")


config.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


[{'label': 'LABEL_2', 'score': 0.9701986312866211}]